# Prepare L.L.Con. Corpus (XML -> SQLite)

This notebook parses the XML corpus under `data/Wendel_Korpus_BVerfG/xml` and
extracts each `<absatz>` as a searchable unit. The output is a SQLite database
with an FTS5 index to support boolean and proximity queries.

Top-level annotation categories (per documentation):
- bverfgdokument (root)
- entscheidung (decision)
- leitsaetze (headnotes)
- rubrum (caption)
- tenor (judgment)
- gruende (reasons)
- abwmeinungen (separate opinions)
- fnentscheidung / fnabwmeinungen (footnotes, older decisions)


In [1]:
from pathlib import Path
from datetime import datetime
import csv
import json
import re
import sqlite3
from lxml import etree

def resolve_corpus_dir() -> Path:
    cwd = Path.cwd()
    for base in [cwd, *cwd.parents]:
        candidate = base / "data" / "Wendel_Korpus_BVerfG" / "xml"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find data/Wendel_Korpus_BVerfG/xml")

CORPUS_DIR = resolve_corpus_dir()
REPO_ROOT = CORPUS_DIR.parents[2]
OUTPUT_DIR = REPO_ROOT / "data_processed"
DB_PATH = OUTPUT_DIR / "corpus.sqlite"
METADATA_PATH = REPO_ROOT / "data" / "Wendel_Korpus_BVerfG" / "Metadaten2.7.1.csv"

TOP_LEVEL_SECTIONS = {
    "leitsaetze",
    "rubrum",
    "tenor",
    "gruende",
    "abwmeinungen",
    "fnentscheidung",
    "fnabwmeinungen",
}

def _clean_value(value: str | None) -> str | None:
    if value is None:
        return None
    cleaned = value.strip()
    if cleaned == "" or cleaned.upper() == "NA":
        return None
    return cleaned

def load_metadata(path: Path) -> dict[str, dict[str, str | int | None]]:
    if not path.exists():
        raise FileNotFoundError(f"Metadata file not found: {path}")
    metadata: dict[str, dict[str, str | int | None]] = {}
    with path.open("r", encoding="utf-8") as handle:
        reader = csv.DictReader(handle, delimiter='	')
        for row in reader:
            key = _clean_value(row.get("dateiname"))
            if not key:
                continue
            year_raw = _clean_value(row.get("jahr"))
            year = int(year_raw) if year_raw and year_raw.isdigit() else None
            metadata[key] = {
                "year": year,
                "band": _clean_value(row.get("band")),
                "entscheidungsart": _clean_value(row.get("entscheidungsart")),
            }
    return metadata

METADATA_BY_ID = load_metadata(METADATA_PATH)


In [2]:
def normalize_text(text: str) -> str:
    cleaned = re.sub(r"\s+", " ", text).strip()
    return cleaned.lower()


def get_section(element: etree._Element) -> str | None:
    for ancestor in element.iterancestors():
        tag = ancestor.tag
        if tag in TOP_LEVEL_SECTIONS:
            return tag
    return None


def get_section_path(element: etree._Element) -> str:
    tags = [element.tag]
    for ancestor in element.iterancestors():
        tags.append(ancestor.tag)
    return "/".join(reversed(tags))


def get_nearest_ebene1(element: etree._Element) -> etree._Element | None:
    for ancestor in element.iterancestors():
        if ancestor.tag == "ebene1":
            return ancestor
    return None


def extract_absatz_records(file_path: Path):
    absatz_index = 0

    for _, element in etree.iterparse(
        str(file_path), events=("end",), tag="absatz", recover=True, huge_tree=True
    ):
        absatz_index += 1
        text_raw = "".join(element.itertext())
        section = get_section(element)
        section_path = get_section_path(element)
        ebene1 = get_nearest_ebene1(element)
        ebene1_attrs = dict(ebene1.attrib) if ebene1 is not None else {}
        metadata = METADATA_BY_ID.get(file_path.stem, {})

        record = {
            "file_name": file_path.name,
            "absatz_index": absatz_index,
            "section": section,
            "section_path": section_path,
            "absatz_id": element.get("absatzID"),
            "rn": element.get("rn"),
            "tbeg": element.get("tbeg"),
            "absatz_ebene1nr": element.get("ebene1nr"),
            "absatz_ebene1zeichen": element.get("ebene1zeichen"),
            "ebene1_nr": ebene1_attrs.get("nr"),
            "ebene1_zeichen": ebene1_attrs.get("zeichen"),
            "ebene1_tbeg": ebene1_attrs.get("tbeg"),
            "year": metadata.get("year"),
            "band": metadata.get("band"),
            "entscheidungsart": metadata.get("entscheidungsart"),
            "text_raw": text_raw,
            "text_norm": normalize_text(text_raw),
            "absatz_attrs_json": json.dumps(
                dict(element.attrib), ensure_ascii=True, sort_keys=True
            ),
            "ebene1_attrs_json": json.dumps(
                ebene1_attrs, ensure_ascii=True, sort_keys=True
            ),
        }

        yield record

        element.clear()
        while element.getprevious() is not None:
            parent = element.getparent()
            if parent is None:
                break
            del parent[0]


In [3]:
OUTPUT_DIR.mkdir(exist_ok=True)
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("PRAGMA synchronous=NORMAL;")

conn.execute("""
CREATE TABLE absatz (
    id INTEGER PRIMARY KEY,
    file_name TEXT NOT NULL,
    absatz_index INTEGER NOT NULL,
    section TEXT,
    section_path TEXT,
    absatz_id TEXT,
    rn TEXT,
    tbeg TEXT,
    absatz_ebene1nr TEXT,
    absatz_ebene1zeichen TEXT,
    ebene1_nr TEXT,
    ebene1_zeichen TEXT,
    ebene1_tbeg TEXT,
    year INTEGER,
    band TEXT,
    entscheidungsart TEXT,
    text_raw TEXT,
    text_norm TEXT,
    absatz_attrs_json TEXT,
    ebene1_attrs_json TEXT
);
""")

conn.execute("CREATE INDEX idx_absatz_file ON absatz(file_name);")
conn.execute("CREATE INDEX idx_absatz_section ON absatz(section);")


In [4]:
files = sorted(CORPUS_DIR.glob("*.xml"))
if not files:
    raise FileNotFoundError(f"No XML files found in {CORPUS_DIR}")
total_absatz = 0
section_counts: dict[str, int] = {}
file_sections: dict[str, set[str]] = {}
sample_rows = []

columns = [
    "file_name",
    "absatz_index",
    "section",
    "section_path",
    "absatz_id",
    "rn",
    "tbeg",
    "absatz_ebene1nr",
    "absatz_ebene1zeichen",
    "ebene1_nr",
    "ebene1_zeichen",
    "ebene1_tbeg",
    "year",
    "band",
    "entscheidungsart",
    "text_raw",
    "text_norm",
    "absatz_attrs_json",
    "ebene1_attrs_json",
]

insert_sql = (
    "INSERT INTO absatz ("
    + ', '.join(columns)
    + ") VALUES ("
    + ', '.join(["?"] * len(columns))
    + ")"
)

batch = []
for file_path in files:
    file_name = file_path.name
     # print(f"Processing {file_name}...")
    for record in extract_absatz_records(file_path):
        total_absatz += 1
        section = record["section"] or "unknown"
        section_counts[section] = section_counts.get(section, 0) + 1
        file_sections.setdefault(file_name, set()).add(section)

        if len(sample_rows) < 5:
            sample_rows.append(record)

        batch.append(tuple(record[col] for col in columns))
        if len(batch) >= 1000:
            conn.executemany(insert_sql, batch)
            batch.clear()

if batch:
    conn.executemany(insert_sql, batch)
    batch.clear()

conn.execute("""
CREATE VIRTUAL TABLE absatz_fts USING fts5(
    text_norm,
    content='absatz',
    content_rowid='id',
    tokenize='unicode61'
);
""")
conn.execute("INSERT INTO absatz_fts(rowid, text_norm) SELECT id, text_norm FROM absatz;")
conn.commit()


In [5]:
print(f"XML files processed: {len(files)}")
print(f"Absatz rows extracted: {total_absatz}")
metadata_hits = sum(1 for file in files if file.stem in METADATA_BY_ID)
print(f"Metadata matches: {metadata_hits}")

print("\nTop sections (by count):")
for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {section}: {count}")

print("\nSample rows:")
sample_rows[:5]

print("\nSample file to section mappings:")
example_mappings = [(k, sorted(v)) for k, v in list(file_sections.items())[:5]]
example_mappings

summary_path = OUTPUT_DIR / "prepare_corpus_summary.txt"
with summary_path.open("w", encoding="utf-8") as handle:
    handle.write("L.L.Con. corpus preprocessing summary\n")
    handle.write(f"Timestamp: {datetime.now().isoformat()}\n")
    handle.write(f"Corpus dir: {CORPUS_DIR}\n")
    handle.write(f"Database: {DB_PATH}\n")
    handle.write(f"XML files processed: {len(files)}\n")
    handle.write(f"Absatz rows extracted: {total_absatz}\n")
    handle.write(f"Metadata matches: {metadata_hits}\n")
    handle.write("Top sections (by count):\n")
    for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        handle.write(f"  {section}: {count}\n")
    handle.write("Sample file to section mappings:\n")
    for file_name, sections in example_mappings:
        handle.write(f"  {file_name}: {', '.join(sections)}\n")
print(f"\nWrote summary log to {summary_path}")


XML files processed: 10980
Absatz rows extracted: 403159
Metadata matches: 10980

Top sections (by count):
  gruende: 383487
  abwmeinungen: 5764
  rubrum: 5339
  tenor: 4689
  leitsaetze: 3880

Sample rows:

Sample file to section mappings:

Wrote summary log to /Volumes/storage/HELPOTHERS/Project_LLCon/data_processed/prepare_corpus_summary.txt


In [6]:
sample_rows[:5]

[{'file_name': 'BVerfGE1,001.xml',
  'absatz_index': 1,
  'section': 'rubrum',
  'section_path': 'bverfgdokument/entscheidung/rubrum/absatz',
  'absatz_id': None,
  'rn': 'NA',
  'tbeg': None,
  'absatz_ebene1nr': None,
  'absatz_ebene1zeichen': None,
  'ebene1_nr': None,
  'ebene1_zeichen': None,
  'ebene1_tbeg': None,
  'year': 1951,
  'band': '1',
  'entscheidungsart': 'Beschluss',
  'text_raw': '\n    Beschluß des Zweiten Senats vom 9. September 1951 - 2 BvQ 1/51 - in dem Verfassungsrechtsstreit betreffend das Zweite Gesetz über die Neugliederung in den Ländern Baden, Württemberg-Baden und Württemberg-Hohenzollern vom 4. Mai 1951 (BGBl. I S. 284); - Antragsteller: Die Badische Landesregierung; Beteiligte: Die Regierung des Landes Württemberg-Baden, die Regierung des Landes Württemberg-Hohenzollern.\n   ',
  'text_norm': 'beschluß des zweiten senats vom 9. september 1951 - 2 bvq 1/51 - in dem verfassungsrechtsstreit betreffend das zweite gesetz über die neugliederung in den ländern 

In [7]:
example_mappings

[('BVerfGE1,001.xml', ['gruende', 'rubrum', 'tenor']),
 ('BVerfGE1,003.xml', ['gruende', 'leitsaetze', 'rubrum', 'tenor']),
 ('BVerfGE1,004.xml', ['gruende', 'leitsaetze', 'rubrum', 'tenor']),
 ('BVerfGE1,005.xml', ['gruende', 'leitsaetze', 'rubrum', 'tenor']),
 ('BVerfGE1,007.xml', ['gruende', 'leitsaetze', 'rubrum', 'tenor'])]